# AI PCB Autorouter Research Platform
## Milestone 1: Single-Net Reinforcement Learning Router (`PCBRouterNet`)

This notebook trains a reinforcement learning spatial planning agent on a **10-channel 256x256 multi-layer PCB grid** without PNS push/shove heuristics.

### Architecture:
- **State**: $(10, 256, 256)$ multi-channel spatial tensor (Copper, Obstacles, Pads, Heads, Congestion Heatmaps, Clearance Cost, Target Distance Field, Layer Occupancy).
- **Actions**: 96 discrete growth decisions (8 compass directions $\times$ 3 step distances $\times$ 2 layers $\times$ 2 vias).
- **Model**: `PCBRouterNet` (Multi-scale CNN patch embedding $\rightarrow$ Transformer Encoder $\rightarrow$ Policy & Value heads).
- **Objective**: Achieve $>95\%$ single-net routing completion rate.

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
# Cell 2: Sync Repository & Install Requirements
if [ -d "/content/Routerv3" ]; then
    cd /content/Routerv3 && git fetch origin main && git reset --hard origin/main
else
    git clone https://github.com/Klutzhehe/Routerv3.git /content/Routerv3
fi
pip install -q gymnasium scipy
echo "=== Repository synced! Current Git commit: ==="
git -C /content/Routerv3 log -n 1 --oneline

In [ ]:
# Cell 3: Instantiate PCBRouterEnv & Verify 10-Channel Observation
import sys
sys.path.insert(0, "/content/Routerv3")

import torch
import numpy as np
from pcbworld.environment import PCBRouterEnv
from pcbworld.renderer import render_grid_board
from IPython.display import Image, display

env = PCBRouterEnv(grid_size=256, num_nets=1, num_obstacles=0, seed=42)
obs, info = env.reset()
print(f"✓ Observation Shape: {obs.shape} (Channels: 10, Height: 256, Width: 256)")
print(f"✓ Action Space: {env.action_space.n} Discrete Actions")
print(f"✓ Initial Head Position: {info['head_pos']}")

In [ ]:
# Cell 4: 🎨 Visualize Board Grid State
fig = render_grid_board(
    copper_grid=env.board.copper_grid,
    pads=[net.source_pad for net in env.board.nets] + [net.target_pad for net in env.board.nets],
    obstacles=env.board.obstacles,
    heads=[{"net_id": 1, "x": env.head_x, "y": env.head_y, "layer": env.head_layer}],
    congestion_map=env._congestion_cache,
    save_path="/content/initial_board_viz.png",
    title="Milestone 1: Single-Net Initial State",
)
display(Image("/content/initial_board_viz.png"))

In [ ]:
# Cell 5: Initialize PCBRouterNet (CNN + Transformer Backbone)
from models.router_policy import PCBRouterNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PCBRouterNet(in_channels=10, action_dim=96, d_model=256, num_transformer_layers=2, num_heads=4).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Model Initialized on {str(device).upper()} with {total_params:,} parameters")

In [ ]:
# Cell 6: 🚀 Run Training Loop for Milestone 1
from training.train import train_single_net_policy

trained_model = train_single_net_policy(
    total_timesteps=30_000,
    rollout_steps=512,
    epochs=4,
    minibatch_size=64,
    lr=3e-4,
    checkpoint_dir="/content/drive/MyDrive/pcb_ai_router/checkpoints",
    device_str="cuda" if torch.cuda.is_available() else "cpu",
)

In [ ]:
# Cell 7: 📊 Display Training Curves and Benchmark Metrics
from training.evaluation import evaluate_policy

curves_path = "/content/drive/MyDrive/pcb_ai_router/checkpoints/single_net_training_curves.png"
if os.path.exists(curves_path):
    display(Image(curves_path))

metrics = evaluate_policy(
    trained_model,
    num_eval_episodes=50,
    num_nets=1,
    num_obstacles=0,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print(f"Final Benchmark Success Rate: {metrics['completion_rate']:.1f}%")